In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


class AutoLoader:
    
    def __init__(
        self,
        source_path: str,
        target_table: str,
        checkpoint_path: str,
        schema_location: str,
        file_format: str = "csv"
    ):
        self.source_path = source_path
        self.target_table = target_table
        self.checkpoint_path = checkpoint_path
        self.schema_location = schema_location
        self.file_format = file_format


    def read_stream(self) -> DataFrame:
        """
        Read incremental files using Databricks Auto Loader.
        """

        df = (
            spark.readStream
                .format("cloudFiles")

                # Source format
                .option("cloudFiles.format", self.file_format)

                # Schema inference/evolution
                .option(
                    "cloudFiles.schemaLocation",
                    self.schema_location
                )
                .option(
                    "cloudFiles.schemaEvolutionMode",
                    "addNewColumns"
                )
                .option(
                    "cloudFiles.inferColumnTypes",
                    "true"
                )

                # CSV specific options
                .option("header", "true")
                .option("delimiter", ",")
                .option("rescuedDataColumn", "_rescued_data")

                .load(self.source_path)
        )

        return df


    def add_metadata(self, df: DataFrame) -> DataFrame:
        """
        Add ingestion/audit metadata to Bronze data.
        """

        return (
            df
            .withColumn(
                "_source_file",
                F.col("_metadata.file_name")
            )
            .withColumn(
                "_source_file_path",
                F.col("_metadata.file_path")
            )
            .withColumn(
                "_source_file_size",
                F.col("_metadata.file_size")
            )
            .withColumn(
                "_source_file_modification_time",
                F.col("_metadata.file_modification_time")
            )
            .withColumn(
                "_ingestion_timestamp",
                F.current_timestamp()
            )
            .withColumn(
                "_ingestion_date",
                F.current_date()
            )
        )


    def write_stream(self, df: DataFrame):
        """
        Write raw streaming data into the Bronze Delta table.
        """

        query = (
            df.writeStream
                .format("delta")
                .option(
                    "checkpointLocation",
                    self.checkpoint_path
                )
                .option(
                    "mergeSchema",
                    "true"
                )
                .outputMode("append")
                .trigger(availableNow=True)
                .toTable(self.target_table)
        )

        return query


    def run(self):
        """
        Complete Bronze ingestion process.
        """

        print(f"Source        : {self.source_path}")
        print(f"Target        : {self.target_table}")
        print(f"Checkpoint    : {self.checkpoint_path}")
        print(f"Schema        : {self.schema_location}")

        df = self.read_stream()

        df = self.add_metadata(df)

        query = self.write_stream(df)

        return query